# fox_agent_core 真实模型实验室

**学习路径：真实模型 → 自定义工具 → 文件工具 → Session → Skill → 手动压缩 → 自动压缩。**

这份 Notebook 配合 [架构学习与面试指南](ARCHITECTURE_GUIDE.md) 使用。每章都有“设计问题 → 实验 → 可观察证据 → 面试复盘”，重点是看懂各层如何协作。

默认沿用已有 `fox_ai/DEEPSEEK_MODEL_LAB.ipynb` 的 DeepSeek 配置，使用 `deepseek-flash` 和 `https://api.deepseek.com`。模型名称可在配置单元修改。**全部模型回答和压缩摘要都来自真实 API，不使用 Faux、预设响应或固定摘要。** 本地练习文件与日志是明确标注的教学数据。

| 章节 | 实际动手内容 | 你应当观察到什么 |
| --- | --- | --- |
| 0 | 环境、密钥、练习目录 | 源码导入与配置，不发请求 |
| 1 | 普通 Agent 请求 | 一次真实响应及生命周期事件 |
| 2 | 自定义计算工具 | ToolCall → 本地执行 → ToolResult → 再次回答 |
| 3 | Harness 文件修改 | 模型真正读取和编辑练习文件 |
| 4 | 保存、恢复、分支 | 新 Harness 从 JSONL 恢复消息与配置 |
| 5 | 发现、读取、显式调用 Skill | 技能目录、正文和实际文件报告之间的关系 |
| 6 | 读取较长日志，手动压缩 | 摘要模型请求、切割点、历史与上下文区别 |
| 7 | 降低本地预算，自动压缩 | 第一次模型请求前先生成摘要 |
| 8 | 用量和复盘 | 请求次数、摘要次数、学习检查表 |

**运行方式**：项目根目录执行 `uv sync`，在 VS Code 中选择项目 `.venv` 的 Python 3.14 内核，然后从上到下运行。代码使用 Notebook 顶层 `await`，不需要 `asyncio.run()`。

每次运行准备单元会创建独立临时目录，模型仅获得该目录下练习文件的读写工具。请求会消耗真实 API 用量；实验有单次轮数与总请求数限制。密钥仅从环境变量、根目录 `.env` 或隐藏输入读取，不放进 Notebook 源码和输出。

## 0. 环境与真实模型配置

先建立三个边界：`fox_ai` 负责请求模型，`Agent` 负责循环与状态，`AgentHarness` 负责 Session、Skill 和 Compaction。

本实验统一从 `fox_ai.src` 导入，避免与 `packages.fox_ai.src` 混用而产生两套 Python 类型。更换导入方式后请重启内核。

In [1]:
import sys
import importlib.util
from pathlib import Path

if sys.version_info < (3, 14):
    raise RuntimeError("请选择项目 Python 3.14+ 内核，然后重新从第一格运行。")
required = ["openai", "anthropic", "pydantic", "jsonschema", "yaml", "IPython"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"当前内核缺少 {missing}；请在项目根目录运行 uv sync 并切换到 .venv 内核。")

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "packages/fox_agent_core/agent.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("请从 FoxCode 仓库内打开 Notebook。")
if str(REPO_ROOT / "packages") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "packages"))

import asyncio
import getpass
import html
import json
import os
import re
import tempfile
import time
from collections import Counter
from contextvars import ContextVar
from dataclasses import asdict
from IPython.display import HTML, Markdown, display

from fox_ai.src import (
    Model, AssistantMessage, UserMessage, ToolCall, ToolResultMessage,
    TextContent, stream_simple,
)
from fox_agent_core import (
    Agent, AgentOptions, AgentHarness, AgentHarnessOptions, AgentToolResult,
    FunctionTool, ReadTool, WriteTool, EditTool,
    Session, JsonlSessionStorage, CompactionSettings,
    LoadSkillsOptions, load_skills, format_skills_for_prompt,
    estimate_tokens, estimate_context_tokens, find_cut_point, should_compact,
)
from fox_agent_core.harness.compaction import SUMMARIZATION_SYSTEM_PROMPT

print("Python:", sys.version.split()[0])
print("Agent 源码:", REPO_ROOT / "packages/fox_agent_core")

Python: 3.14.7
Agent 源码: c:\Users\Qin\Desktop\coding_agent\FoxCode\packages\fox_agent_core


### 配置中哪些数字代表真实服务，哪些是教学预算？

- `MODEL_ID` / `BASE_URL` 是真实请求目的地，默认与已有 DeepSeek 实验一致。
- `LAB_CONTEXT_WINDOW` 是本实验交给 Harness 的**本地上下文预算**，不宣称它等于服务商实际最大窗口。
- `OUTPUT_TOKENS` 控制单次回答的输出上限；`MAX_REAL_REQUESTS` 限制本次 Notebook 的模型请求总数，包含摘要调用。
- 本实验关闭 DeepSeek thinking，以便集中观察 Agent 架构；改成其他兼容服务时，应相应调整 `sampling_params`。

读取 `.env` 只匹配指定的一个密钥名，不打印文件内容，也不把整份环境配置传给模型。找不到时才出现隐藏输入框。

In [2]:
MODEL_ID = os.getenv("FOX_AGENT_MODEL_ID", "deepseek-flash")
BASE_URL = os.getenv("FOX_AGENT_BASE_URL", "https://api.deepseek.com")
API_KEY_ENV = "DEEPSEEK_API_KEY"
LAB_CONTEXT_WINDOW = 32768       # 本地教学预算，不是对服务商规格的声明
OUTPUT_TOKENS = 1200
MAX_REAL_REQUESTS = 32

def load_key(name):
    value = os.environ.get(name, "").strip()
    env_file = REPO_ROOT / ".env"
    if not value and env_file.is_file():
        for line in env_file.read_text(encoding="utf-8-sig").splitlines():
            left, sep, right = line.strip().removeprefix("export ").partition("=")
            if sep and left.strip() == name:
                value = right.strip()
                if len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"'):
                    value = value[1:-1]
                else:
                    value = value.split(" #", 1)[0].strip()
                break
    return value or getpass.getpass(f"请输入 {name}（不回显）：").strip()

API_KEY = load_key(API_KEY_ENV)
if not API_KEY:
    raise RuntimeError("没有可用密钥。请设置 DEEPSEEK_API_KEY 后重跑此单元。")

model = Model(
    id=MODEL_ID, name="真实模型 · Agent 学习实验", provider="deepseek",
    api="openai-completions", base_url=BASE_URL, input=["text"],
    reasoning=False, context_window=LAB_CONTEXT_WINDOW, max_tokens=OUTPUT_TOKENS,
    compat={"supportsStrictMode": False},
)
STREAM_OPTIONS = {
    "api_key": API_KEY, "max_tokens": OUTPUT_TOKENS,
    "timeout_ms": 90000, "max_retries": 0,
    "sampling_params": {"extra_body": {"thinking": {"type": "disabled"}}},
}
print({"model": model.id, "api": model.api, "base_url": model.base_url,
       "local_context_budget": model.context_window, "key_loaded": bool(API_KEY)})

{'model': 'deepseek-flash', 'api': 'openai-completions', 'base_url': 'https://api.deepseek.com', 'local_context_budget': 32768, 'key_loaded': True}


### 一组观察辅助函数（可以折叠）

下面的代码只负责记录请求和渲染事件。`observed_stream` **始终调用真实的 `stream_simple`**；包装器不会改写模型回答，也不会替代摘要。

观察表记录三类信息：模型请求及 token 用量、Agent 生命周期事件、完成消息。不要打印 `STREAM_OPTIONS`，因为其中包含密钥。

In [3]:
PHASE = ContextVar("fox_agent_lab_phase", default="未标记")
REQUESTS = []
EVENTS = []
ATTACHED = {}

def safe_text(value):
    text = str(value).replace(API_KEY, "[REDACTED]")
    return re.sub(r"sk-[A-Za-z0-9_-]{12,}", "[REDACTED]", text)

def table(rows):
    if not rows:
        print("（没有记录）")
        return
    columns = list(rows[0])
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(safe_text(row.get(c, '')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

class MeasuredStream:
    def __init__(self, inner, row):
        self.inner, self.row = inner, row

    def __aiter__(self):
        return self.inner.__aiter__()

    async def result(self):
        message = await self.inner.result()
        usage = message.usage
        self.row.update({
            "status": message.stop_reason,
            "seconds": round(time.perf_counter() - self.row["started"], 2),
            "input": usage.input, "output": usage.output,
            "cache_read": usage.cache_read, "cache_write": usage.cache_write,
            "total": usage.total_tokens or (usage.input + usage.output + usage.cache_read + usage.cache_write),
        })
        return message

    async def aclose(self):
        await self.inner.aclose()
        if self.row["status"] == "pending":
            self.row["status"] = "closed"

def observed_stream(request_model, context, options):
    if len(REQUESTS) >= MAX_REAL_REQUESTS:
        raise RuntimeError("达到本次 Notebook 的真实请求上限；检查用量后再调整 MAX_REAL_REQUESTS。")
    row = {
        "number": len(REQUESTS) + 1, "phase": PHASE.get(),
        "kind": "summary" if context.system_prompt == SUMMARIZATION_SYSTEM_PROMPT else "agent",
        "model": request_model.id, "messages": len(context.messages),
        "estimated_messages": estimate_context_tokens(context.messages),
        "tools": [t.name for t in context.tools or []],
        "started": time.perf_counter(), "status": "pending",
    }
    REQUESTS.append(row)
    try:
        return MeasuredStream(stream_simple(request_model, context, options), row)
    except Exception:
        row["status"] = "request_setup_error"
        raise

def message_text(message):
    if isinstance(message.content, str):
        return message.content
    return "\n".join(c.text for c in message.content if isinstance(c, TextContent))

def final_text(agent_or_harness):
    last = next((m for m in reversed(agent_or_harness.state.messages)
                 if isinstance(m, AssistantMessage)), None)
    return message_text(last) if last else ""

def attach(subject, label):
    old = ATTACHED.pop(id(subject), None)
    if old:
        old()

    def on_event(event, cancel_event):
        row = {"phase": PHASE.get(), "owner": label, "type": event.type}
        if event.type.startswith("tool_execution"):
            row.update({"tool": event.tool_name, "call_id": event.tool_call_id})
        if event.type == "tool_execution_start":
            row["args"] = event.args
            print(f"\n[工具开始] {event.tool_name} {safe_text(event.args)}")
        elif event.type == "tool_execution_end":
            row["is_error"] = event.is_error
            print(f"[工具结束] {event.tool_name} / error={event.is_error}")
        elif event.type == "message_update":
            delta = event.assistant_message_event
            if delta.type == "text_delta":
                print(safe_text(delta.delta), end="", flush=True)
        elif event.type in ("turn_start", "agent_end", "compaction_start", "compaction_end", "compaction_error"):
            print(f"\n[{event.type}]")
        EVENTS.append(row)

    ATTACHED[id(subject)] = subject.subscribe(on_event)

async def run_step(subject, text, phase, *, skill=None):
    token = PHASE.set(phase)
    start = len(REQUESTS)
    try:
        operation = subject.invoke_skill(skill, text) if skill else subject.prompt(text)
        await asyncio.wait_for(operation, timeout=240)
        if subject.state.error_message:
            raise RuntimeError(safe_text(subject.state.error_message))
        print(f"\n本步骤新增真实请求：{len(REQUESTS) - start}")
        return final_text(subject)
    except Exception as exc:
        raise RuntimeError(safe_text(exc)) from None
    finally:
        PHASE.reset(token)

async def run_compaction(subject, phase):
    token = PHASE.set(phase)
    try:
        return await asyncio.wait_for(subject.compact(), timeout=180)
    except Exception as exc:
        raise RuntimeError(safe_text(exc)) from None
    finally:
        PHASE.reset(token)

def show_messages(subject, limit=14):
    messages = subject.state.messages
    rows = []
    for index, message in enumerate(messages[-limit:], max(0, len(messages) - limit)):
        calls = [c.name for c in message.content if isinstance(c, ToolCall)] if isinstance(message.content, list) else []
        rows.append({"index": index, "role": message.role,
                     "calls": ", ".join(calls), "call_id": getattr(message, "tool_call_id", ""),
                     "preview": message_text(message)[:120]})
    table(rows)

def assert_paired(messages):
    waiting = set()
    for message in messages:
        if isinstance(message, AssistantMessage):
            assert not waiting, f"上一轮仍有未配对工具调用：{waiting}"
            waiting = {c.id for c in message.content if isinstance(c, ToolCall)}
        elif isinstance(message, ToolResultMessage):
            assert message.tool_call_id in waiting, "出现没有对应调用的工具结果"
            waiting.remove(message.tool_call_id)
        elif isinstance(message, UserMessage):
            assert not waiting, "用户消息插入到了未完成的工具调用组中"
    assert not waiting, f"存在未完成工具调用：{waiting}"

## 1. 一次真实请求：Agent 层先提供什么？

此处只使用 `Agent`，没有 Session、Skill 和压缩。先观察 `agent_start → turn_start → message_* → turn_end → agent_end`。

**运行前预测**：没有工具、没有排队消息时，会请求模型几次？回答保存到哪里？

In [4]:
basic = Agent(AgentOptions(
    initial_state={"model": model, "system_prompt": "请用简洁中文回答。", "tools": []},
    stream_fn=observed_stream, stream_options=STREAM_OPTIONS, max_turns=3,
))
attach(basic, "basic")
before = len(REQUESTS)
await run_step(basic, "用一句话说明：为什么工具执行后还需要把结果交给模型？", "01-普通响应")
assert len(REQUESTS) == before + 1
assert not basic.is_running
show_messages(basic)


[turn_start]
工具只负责执行，模型负责理解和决策，把结果交给模型才能让它据此判断下一步该做什么。
[agent_end]

本步骤新增真实请求：1


index,role,calls,call_id,preview
0,user,,,用一句话说明：为什么工具执行后还需要把结果交给模型？
1,assistant,,,工具只负责执行，模型负责理解和决策，把结果交给模型才能让它据此判断下一步该做什么。


**观察与面试复盘**：模型请求结束不需要再次轮询，因为没有新工作。Agent 仍保存了用户和助手两条消息，说明 Agent 是有状态对象；它尚未把历史写到磁盘。

## 2. 自定义工具：模型提出调用，Python 负责执行

我们提供一个 `capacity_budget` 工具，计算 `workers × 每个 worker 的并发上限`。这里的计算只是教学例子，不是生产容量评估公式。

关键不是乘法，而是三段边界：**Schema 声明 → 运行时校验和本地执行 → ToolResult 回传模型**。不在提示词中预先提供工具结果。

In [5]:
EXECUTED_BUDGETS = []

async def capacity_budget(call_id, params, cancel_event, on_update):
    result = params["workers"] * params["concurrency_per_worker"]
    EXECUTED_BUDGETS.append({"call_id": call_id, **params, "result": result})
    return AgentToolResult(
        content=[TextContent(text=json.dumps({"concurrency_budget": result}))],
        details={"formula": "workers * concurrency_per_worker"},
    )

budget_tool = FunctionTool(
    name="capacity_budget", description="计算本练习的总并发额度；必须用此工具获取结果。",
    parameters={"type": "object", "properties": {
        "workers": {"type": "integer", "minimum": 1},
        "concurrency_per_worker": {"type": "integer", "minimum": 1},
    }, "required": ["workers", "concurrency_per_worker"], "additionalProperties": False},
    handler=capacity_budget,
)
tool_agent = Agent(AgentOptions(
    initial_state={"model": model, "tools": [budget_tool],
                   "system_prompt": "你正在演示工具闭环。计算题必须调用 capacity_budget，不能自行给出结果。"},
    stream_fn=observed_stream, stream_options=STREAM_OPTIONS, max_turns=4,
))
attach(tool_agent, "custom-tool")
await run_step(tool_agent, "请调用 capacity_budget：workers=4，每个 worker 的并发上限为 37，然后解释工具给出的结果。", "02-自定义工具")
assert EXECUTED_BUDGETS, "模型没有调用工具，请检查端点是否支持 tool calling，并重跑本实验。"
assert EXECUTED_BUDGETS[-1]["result"] == 148
assert_paired(tool_agent.state.messages)
table(EXECUTED_BUDGETS)
show_messages(tool_agent)


[turn_start]

[工具开始] capacity_budget {'workers': 4, 'concurrency_per_worker': 37}
[工具结束] capacity_budget / error=False

[turn_start]
工具已调用完成，结果如下：

- **总并发额度（concurrency_budget）：148**

解释：该额度等于 workers 数量乘以每个 worker 的并发上限，即 4 × 37 = 148。也就是说，在 4 个 worker、每个最多同时处理 37 个任务的前提下，本练习允许的并发总量为 148。这个数字是由 capacity_budget 工具返回的权威结果，请以此为准。
[agent_end]

本步骤新增真实请求：2


call_id,workers,concurrency_per_worker,result
call_00_hFu9OZkfdH1DsHiRRJ3G3678,4,37,148


index,role,calls,call_id,preview
0,user,,,请调用 capacity_budget：workers=4，每个 worker 的并发上限为 37，然后解释工具给出的结果。
1,assistant,capacity_budget,,
2,toolResult,,call_00_hFu9OZkfdH1DsHiRRJ3G3678,"{""concurrency_budget"": 148}"
3,assistant,,,工具已调用完成，结果如下： - **总并发额度（concurrency_budget）：148** 解释：该额度等于 workers 数量乘以每个 worker 的并发上限，即 4 × 37 = 148。也就是说，在 4 个 worke


**应看到的证据**：`EXECUTED_BUDGETS` 有本地执行记录；消息顺序包含 assistant 工具调用和 toolResult；随后才是模型解释。一次 `prompt` 因此包含多个 turn。

**面试追问**：如果模型传入字符串而非整数，应该在哪一层拒绝？如果工具名称相同但调用两次，怎样区分结果？答案分别是执行前的 Schema 校验，以及 `tool_call_id`。

## 3. Harness：文件工具和持久化一起工作

现在创建一个独立练习目录，里面只有教学配置和日志。会话 JSONL 放在目录外侧的 `sessions/`，模型只能读写内侧 `workspace/`。

教学限制通过 `before_tool_call` 实现路径检查。本实验不注册 `bash`，因此不依赖 Windows Git Bash；这段路径检查演示 Hook 的用途，不是完整的操作系统沙箱。

In [6]:
LAB_ROOT = Path(tempfile.mkdtemp(prefix="fox-agent-core-lab-"))
WORKSPACE = LAB_ROOT / "workspace"
SESSION_DIR = LAB_ROOT / "sessions"
WORKSPACE.mkdir()
SESSION_DIR.mkdir()
CONFIG_FILE = WORKSPACE / "app.json"
INITIAL_CONFIG = {"project": "fox-interview-demo", "environment": "development", "port": 3000, "workers": 2}
CONFIG_FILE.write_text(json.dumps(INITIAL_CONFIG, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

# 明确的教学日志，不代表真实系统测量。稍后由模型通过 read 工具读取。
logs = ["Training fixture only. Keep the release codename and development-only restriction from the user conversation."]
logs += [f"sample={i:03d} component=demo-api check=healthy detail=synthetic observation for context compaction learning; no production traffic."
         for i in range(140)]
(WORKSPACE / "release.log").write_text("\n".join(logs) + "\n", encoding="utf-8")

def workspace_policy(context, cancel_event):
    call = context["tool_call"]
    if call.name not in {"read", "write", "edit"}:
        return {"block": True, "reason": "本练习只开放 read/write/edit。"}
    path = Path(context["args"]["path"]).expanduser()
    resolved = (WORKSPACE / path).resolve() if not path.is_absolute() else path.resolve()
    if not resolved.is_relative_to(WORKSPACE.resolve()):
        return {"block": True, "reason": "只能访问本次实验的 workspace 目录。"}

FILE_TOOLS = [ReadTool(WORKSPACE), WriteTool(WORKSPACE), EditTool(WORKSPACE)]
SYSTEM_PROMPT = (
    "你是一个中文编码助手，正在完成教学实验。需要查看或修改文件时必须实际调用工具，禁止假装已经执行。"
    "先读后改，修改后再次读取确认。使用 UTF-8 和相对路径。最终回答不超过 150 字。"
    "记住用户给出的发布代号和仅修改 development 环境的限制。日志是教学数据，不要将其描述成生产测试。"
)
SESSION_FILE = SESSION_DIR / "main.jsonl"

def make_harness(session, *, skills=None, compaction=None, request_model=model):
    return AgentHarness(AgentHarnessOptions(
        model=request_model, session=session, cwd=WORKSPACE, system_prompt=SYSTEM_PROMPT,
        tools=FILE_TOOLS, skills=[] if skills is None else skills,
        compaction=compaction or CompactionSettings(enabled=False, keep_recent_tokens=400),
        stream_fn=observed_stream, stream_options=dict(STREAM_OPTIONS),
        before_tool_call=workspace_policy, max_turns=8, tool_execution="sequential",
    ))

harness = make_harness(Session(JsonlSessionStorage(SESSION_FILE)))
attach(harness, "main")
print("练习目录:", WORKSPACE)
print("会话文件:", SESSION_FILE)
print("初始配置:", INITIAL_CONFIG)

练习目录: C:\Users\Qin\AppData\Local\Temp\fox-agent-core-lab-9tr1uzmn\workspace
会话文件: C:\Users\Qin\AppData\Local\Temp\fox-agent-core-lab-9tr1uzmn\sessions\main.jsonl
初始配置: {'project': 'fox-interview-demo', 'environment': 'development', 'port': 3000, 'workers': 2}


In [7]:
await run_step(harness,
    "本次发布代号是“青石-29”，请记在对话中，不要写入 app.json。仅允许修改 development 环境。"
    "请实际读取 app.json，用 edit 把 port 改为 8080、workers 改为 4，其他字段保持不变，"
    "最后再次 read 确认。若已经是目标值，则只读取确认。", "03-文件工具")

actual = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
assert actual == {**INITIAL_CONFIG, "port": 8080, "workers": 4}, "文件修改没有满足要求，请查看工具事件。"
assert_paired(harness.state.messages)
print("文件验证通过:", actual)
show_messages(harness)


[turn_start]
I'll start by reading the file.
[工具开始] read {'path': 'app.json'}
[工具结束] read / error=False

[turn_start]

[工具开始] edit {'path': 'app.json', 'old_text': '  "port": 3000,\n  "workers": 2', 'new_text': '  "port": 8080,\n  "workers": 4'}
[工具结束] edit / error=True

[turn_start]

[工具开始] edit {'path': 'app.json', 'old_text': '  "port": 3000,\n  "workers": 2\n}', 'new_text': '  "port": 8080,\n  "workers": 4\n}'}
[工具结束] edit / error=True

[turn_start]

[工具开始] read {'path': 'app.json', 'offset': 1, 'limit': 6}
[工具结束] read / error=False

[turn_start]

[工具开始] edit {'path': 'app.json', 'old_text': '"port": 3000', 'new_text': '"port": 8080'}
[工具结束] edit / error=False

[turn_start]

[工具开始] edit {'new_text': '"workers": 4', 'old_text': '"workers": 2', 'path': 'app.json'}
[工具结束] edit / error=False

[turn_start]

[工具开始] read {'path': 'app.json'}
[工具结束] read / error=False

[turn_start]
已完成。发布代号“青石-29”仅记于对话，未写入 app.json。已用 edit 将 development 环境的 port 改为 8080、workers 改为 4，其余字段不变；再次 read 确认结果为 p

index,role,calls,call_id,preview
2,toolResult,,call_00_SAAsCcynzAjlEYDO84CI2715,"1: { 2: ""project"": ""fox-interview-demo"", 3: ""environment"": ""development"", 4: ""port"": 3000, 5: ""workers"": 2 6: }"
3,assistant,edit,,
4,toolResult,,call_00_s6o5UghL7UeKPeCOcjP56499,old_text must match exactly once; read the file and include more context
5,assistant,edit,,
6,toolResult,,call_00_jS0ZttdXzLhwId1cj1Iu1098,old_text must match exactly once; read the file and include more context
7,assistant,read,,
8,toolResult,,call_00_8vqN8S9zWvgkHSbWJpZ89035,"1: { 2: ""project"": ""fox-interview-demo"", 3: ""environment"": ""development"", 4: ""port"": 3000, 5: ""workers"": 2 6: }"
9,assistant,edit,,
10,toolResult,,call_00_XIVGUUVbaI0zVufXzeWQ1005,Edited C:\Users\Qin\AppData\Local\Temp\fox-agent-core-lab-9tr1uzmn\workspace\app.json
11,assistant,edit,,


**观察与复盘**：文件内容由 Python 读取验证，不凭模型一句“修改成功”就判定完成。Harness 每收到一条完成消息就保存它，工具结果也属于正式消息；一次成功文件修改并不等于整个 run 结束，模型还会读取确认并输出说明。

## 4. Session：销毁对象的假设下，怎样继续对话？

下面重新打开 JSONL 并创建新的 Harness，不把旧对象的 `state.messages` 直接传过去。先检查重建结果，再让真实模型回答只出现在历史中的发布代号。

新对象暂时关闭所有工具，避免它从文件里寻找答案。完成后恢复工具集合，供下一章使用。主线按顺序使用新对象，不让两个 Harness 同时写同一个文件。

In [8]:
entries = harness.session.get_entries()
table([{"index": i, "id": e.id[:8], "parent": (e.parent_id or "")[:8], "type": e.type}
       for i, e in enumerate(entries[-16:], max(0, len(entries) - 16))])
before_restore = [m.model_dump(mode="json") for m in harness.state.messages]
restored = make_harness(Session(JsonlSessionStorage(SESSION_FILE)), request_model=None)
attach(restored, "restored")
assert [m.model_dump(mode="json") for m in restored.state.messages] == before_restore
print("重建消息数:", len(restored.state.messages), "恢复模型:", restored.state.model.id)

restored.set_active_tools([])
answer = await run_step(restored, "只根据已有对话回答：本次发布代号是什么，修改被限制在哪个环境？不要调用工具。", "04-恢复后的真实回答")
assert "青石-29" in answer, "真实模型未正确回忆代号，请检查历史和本次响应；不要用固定答案代替模型回答。"
restored.set_active_tools(["read", "write", "edit"])

index,id,parent,type
3,632798e9,6f6fab6a,message
4,e05c2e8f,632798e9,message
5,7ae43643,e05c2e8f,message
6,46571ef2,7ae43643,message
7,c500c2ad,46571ef2,message
8,a2e1502e,c500c2ad,message
9,c058ec7d,a2e1502e,message
10,647b45d1,c058ec7d,message
11,73bf9aec,647b45d1,message
12,a6f94fec,73bf9aec,message


重建消息数: 16 恢复模型: deepseek-flash

[turn_start]
本次发布代号是“青石-29”，修改仅限 development 环境。
[agent_end]

本步骤新增真实请求：1


In [9]:
# 分支操作不需要调用模型：fork 有独立 session ID，消息来自选定历史路径。
branch = restored.fork()
assert branch.session.storage.get_metadata()["id"] != restored.session.storage.get_metadata()["id"]
branch.set_active_tools([])
print("原会话工具:", [t.name for t in restored.state.tools])
print("分支会话工具:", [t.name for t in branch.state.tools])
print("分支消息数:", len(branch.state.messages))

原会话工具: ['read', 'write', 'edit']
分支会话工具: []
分支消息数: 18


**面试复盘**：Session 保存的是消息、配置与分支关系。恢复时重新构建 Agent，不会恢复正在执行的 Python 协程。`fork()` 的结果当前是独立内存会话；JSONL 的逻辑追加使用全文件原子替换实现，并不是只向文件末尾写一行。

## 5. Skill：把任务方法放进上下文，工具仍负责执行

本章在练习目录创建一个 `release-audit` 技能。技能描述告诉模型何时使用，正文说明如何读取配置和输出检查报告。

我们分别观察两条路径：

1. **发现与按需读取**：system prompt 只有目录，模型用 `read` 读取 `SKILL.md`。
2. **显式调用**：`invoke_skill()` 把技能正文直接加入用户消息，不必再通过工具加载正文。

技能中的报告格式是约定，不会自己执行代码；实际读取和写入仍通过 read/write 工具完成。

In [10]:
SKILL_DIR = WORKSPACE / ".fox" / "skills" / "release-audit"
SKILL_DIR.mkdir(parents=True)
SKILL_FILE = SKILL_DIR / "SKILL.md"
SKILL_FILE.write_text("""---
name: release-audit
description: 检查练习项目的开发环境发布配置，并生成结构化验收报告。
---
# 开发环境发布验收

本文中的相对文件路径，除用户指定的工作目录文件外，均相对技能目录解释。
本练习工作目录下的 app.json 是检查对象，audit.md 是报告文件。

只有用户要求实际验收时，执行以下步骤：
1. 使用 read 读取工作目录下的 app.json，禁止根据旧回复猜测内容。
2. 核对 environment=development、port=8080、workers=4；任何不符都写明。
3. 使用 write 写入工作目录下的 audit.md，不修改 app.json。
4. 报告含“检查对象”“检查结果”“注意事项”三个小节。
5. 注意事项必须说明仅检查开发环境，不能把教学日志称为生产测试。
6. 报告末尾独立写一行 AUDIT_LAB_V1，作为本技能的格式验收标记。

如果用户仅要求解释本技能，不执行文件操作，直接解释方法。
""".replace("\n    ", "\n"), encoding="utf-8")

loaded = load_skills(LoadSkillsOptions(
    cwd=str(WORKSPACE), skill_paths=[str(WORKSPACE / ".fox/skills")], include_defaults=False,
))
assert [s.name for s in loaded.skills] == ["release-audit"]
assert not loaded.diagnostics, [asdict(d) for d in loaded.diagnostics]
catalog = format_skills_for_prompt(loaded.skills)
assert "release-audit" in catalog and "AUDIT_LAB_V1" not in catalog
print(catalog)

# 重建宿主，使发现的技能目录进入 system prompt；沿用同一个持久会话。
skilled = make_harness(Session(JsonlSessionStorage(SESSION_FILE)), skills=loaded.skills, request_model=None)
attach(skilled, "with-skill")

The following skills provide specialized instructions for specific tasks.
Read the full skill file when the task matches its description.
When a skill file references a relative path, resolve it against the skill directory (parent of SKILL.md / dirname of the path).

<available_skills>
  <skill>
    <name>release-audit</name>
    <description>检查练习项目的开发环境发布配置，并生成结构化验收报告。</description>
    <location>C:\Users\Qin\AppData\Local\Temp\fox-agent-core-lab-9tr1uzmn\workspace\.fox\skills\release-audit\SKILL.md</location>
  </skill>
</available_skills>


In [11]:
event_start = len(EVENTS)
await run_step(skilled,
    "请使用可用技能 release-audit 实际验收开发环境配置。先用 read 加载该技能的 SKILL.md，"
    "再严格按技能读取 app.json 并写出 audit.md，最后说明结果。", "05-Skill按需读取")
phase_events = EVENTS[event_start:]
skill_reads = [e for e in phase_events if e["type"] == "tool_execution_start" and e.get("tool") == "read"
               and "SKILL.md" in e.get("args", {}).get("path", "")]
assert skill_reads, "没有观察到真实的技能文件读取；请查看工具调用与服务的工具支持。"
report_path = WORKSPACE / "audit.md"
assert report_path.is_file(), "模型没有实际写出验收报告。"
report = report_path.read_text(encoding="utf-8")
assert "AUDIT_LAB_V1" in report, "报告没有遵循技能规定的验收标记。"
assert json.loads(CONFIG_FILE.read_text(encoding="utf-8")) == actual
display(Markdown(report))


[turn_start]

[工具开始] read {'path': '.fox/skills/release-audit/SKILL.md'}
[工具结束] read / error=False

[turn_start]

[工具开始] read {'path': 'app.json'}
[工具结束] read / error=False

[turn_start]

[工具开始] write {'path': 'audit.md', 'content': '# 开发环境发布验收报告\n\n发布代号：青石-29（仅记录于对话，未写入 app.json）\n\n## 检查对象\n\n- 文件：工作目录下的 app.json\n- 项目：fox-interview-demo\n- 检查范围：仅 development 环境\n\n## 检查结果\n\n| 字段 | 期望值 | 实际值 | 结果 |\n| --- | --- | --- | --- |\n| environment | development | development | 通过 |\n| port | 8080 | 8080 | 通过 |\n| workers | 4 | 4 | 通过 |\n\n结论：app.json 中开发环境配置全部符合预期，无不符合项。app.json 未被本次验收修改。\n\n## 注意事项\n\n- 本次仅检查开发环境（development），未涉及其他环境。\n- 教学日志属于教学数据，不能将其描述为生产测试。\n- 发布代号“青石-29”未写入 app.json，仅保留在对话记录中。\n\nAUDIT_LAB_V1\n'}
[工具结束] write / error=False

[turn_start]

[工具开始] read {'path': 'audit.md'}
[工具结束] read / error=False

[turn_start]
验收完成：按 SKILL.md 读取 app.json，environment=development、port=8080、workers=4 全部通过，未修改 app.json；audit.md 已含三小节并以 AUDIT_LAB_V1 结尾，已复读确认。发布代号青石-29 仅留在对话。
[agent_end]

本

# 开发环境发布验收报告

发布代号：青石-29（仅记录于对话，未写入 app.json）

## 检查对象

- 文件：工作目录下的 app.json
- 项目：fox-interview-demo
- 检查范围：仅 development 环境

## 检查结果

| 字段 | 期望值 | 实际值 | 结果 |
| --- | --- | --- | --- |
| environment | development | development | 通过 |
| port | 8080 | 8080 | 通过 |
| workers | 4 | 4 | 通过 |

结论：app.json 中开发环境配置全部符合预期，无不符合项。app.json 未被本次验收修改。

## 注意事项

- 本次仅检查开发环境（development），未涉及其他环境。
- 教学日志属于教学数据，不能将其描述为生产测试。
- 发布代号“青石-29”未写入 app.json，仅保留在对话记录中。

AUDIT_LAB_V1


In [12]:
await run_step(skilled,
    "这次只解释方法，不执行文件操作。请用三条要点说明这个技能检查什么，以及 Skill 为什么不等于 Tool。",
    "05-Skill显式调用", skill="release-audit")
last_user = next(m for m in reversed(skilled.state.messages) if isinstance(m, UserMessage))
assert '<skill name="release-audit"' in message_text(last_user)
assert "AUDIT_LAB_V1" in message_text(last_user)
print("显式调用的用户消息已包含技能正文；无需把所有技能全文长期放入 system prompt。")


[turn_start]
1. 检查对象是工作目录下的 app.json，核对 environment=development、port=8080、workers=4，不符须写明。
2. 实际验收时先 read 再 write 生成 audit.md（含“检查对象/检查结果/注意事项”三节，末尾独立一行 AUDIT_LAB_V1），且不修改 app.json；仅解释时不碰文件。
3. Skill 是任务方法与流程说明（怎么做、按什么标准），Tool 是具体执行动作的接口（如 read/write）；前者提供指导，后者真正读写文件。
[agent_end]

本步骤新增真实请求：1
显式调用的用户消息已包含技能正文；无需把所有技能全文长期放入 system prompt。


**观察与复盘**：模型写出的 `audit.md` 是执行证据，`AUDIT_LAB_V1` 是遵循技能正文的证据。目录中不包含这个标记，因此只看名称和描述不够。

`invoke_skill()` 改变的是输入组织方式，而不是绕过工具权限。Skill 也不是独立的 Agent、操作系统权限或执行脚本。

## 6. 手动压缩：缩短下一次请求，不删除原始历史

### 6.1 让真实模型读取较长的教学日志

前面已经积累了真实工具结果和多轮回答。现在用 `read` 读入较长的重复日志，使上下文增长更明显。日志文本是本 Notebook 生成的教学输入，模型的读取决定和回答仍来自真实服务。

只要求模型给出短结论，避免把完整日志再复述一次。此时自动压缩仍关闭，所以可以清楚比较手动压缩前后。

In [13]:
await run_step(skilled,
    "为了学习上下文管理，请实际 read 工作目录下的 release.log，limit=200。"
    "这是明确标注的教学数据，不代表生产测量。读取后用两句话说明其重复性，"
    "不要复述全部日志。继续保留最初的发布代号、配置结果和仅限开发环境的约束。", "06-读取长日志")
assert_paired(skilled.state.messages)
PRE_COMPACT_LEAF = skilled.session.leaf_id
before_messages = list(skilled.state.messages)
before_estimate = estimate_context_tokens(before_messages)
before_entry_count = len(skilled.session.get_entries())
cut = find_cut_point(before_messages, skilled.options.compaction.keep_recent_tokens)
assert cut > 0, "当前没有可总结的前缀；增加历史或减小 keep_recent_tokens 后再实验。"
table([{"messages": len(before_messages), "estimated_tokens": before_estimate,
        "cut_index": cut, "summarized_messages": cut, "retained_messages": len(before_messages) - cut}])
print("保留尾部的首条角色:", before_messages[cut].role)


[turn_start]

[工具开始] read {'path': 'release.log', 'limit': 200}
[工具结束] read / error=False

[turn_start]
已读取 release.log 前 200 行（实际 141 行）：内容高度重复，除首行说明与 sample 编号递增外，每行字段结构完全一致。

均为 component=demo-api、check=healthy 的合成教学数据。发布代号仍为“青石-29”，配置为 development 环境 port=8080、workers=4，修改仅限开发环境。
[agent_end]

本步骤新增真实请求：2


messages,estimated_tokens,cut_index,summarized_messages,retained_messages
34,7247,33,33,1


保留尾部的首条角色: assistant


### 6.2 由真实模型生成摘要，再提交压缩条目

`enabled=False` 只关闭**自动触发**，不禁止手动 `compact()`。切割点会避开正常的工具调用/结果配对边界。

本单元将实际调用模型生成摘要；没有注入自定义 `summary_fn`。只有摘要成功并且非空，Harness 才写入 compaction 条目并重建上下文。

In [14]:
request_start = len(REQUESTS)
compact_result = await run_compaction(skilled, "06-真实摘要手动压缩")
summary_requests = [r for r in REQUESTS[request_start:] if r["kind"] == "summary"]
assert len(summary_requests) == 1, "没有观察到一次真实摘要请求。"
assert compact_result.removed_count > 0 and compact_result.summary.strip()
assert len(skilled.session.get_entries()) == before_entry_count + 1
assert skilled.session.get_entries()[-1].type == "compaction"
assert_paired(skilled.state.messages)

after_estimate = estimate_context_tokens(skilled.state.messages)
table([
    {"阶段": "压缩前", "上下文消息": len(before_messages), "估算token": before_estimate, "历史条目": before_entry_count},
    {"阶段": "压缩后", "上下文消息": len(skilled.state.messages), "估算token": after_estimate,
     "历史条目": len(skilled.session.get_entries())},
])
print("真实摘要（不是预设答案）：")
display(Markdown(compact_result.summary))
if after_estimate >= before_estimate:
    print("本次摘要没有变短：摘要属于模型输出，压缩率不是硬保证。可缩短摘要或增加待总结内容。")


[compaction_start]

[compaction_end]


阶段,上下文消息,估算token,历史条目
压缩前,34,7247,39
压缩后,2,834,40


真实摘要（不是预设答案）：


# 对话摘要

## 背景与约束
- **发布代号**：“青石-29”，仅记录在对话中，**不得写入 app.json**。
- **环境限制**：仅允许修改/检查 development 环境。
- 工作目录：`C:\Users\Qin\AppData\Local\Temp\fox-agent-core-lab-9tr1uzmn\workspace\`
- app.json 项目名：`fox-interview-demo`

## 已完成工作

### 1. 配置修改（app.json）
- 初始读取 app.json：environment=development、port=3000、workers=2。
- 通过 edit 将 port 改为 **8080**、workers 改为 **4**，其他字段保持不变。
- （注：首次两次 edit 因 old_text 未精确匹配失败，第三次按字段单独替换成功。）
- 最终再次 read 确认：
  ```
  environment: development
  port: 8080
  workers: 4
  ```

### 2. 发布代号确认问答
- 确认发布代号为“青石-29”，修改仅限 development 环境（未调用工具）。

### 3. 技能 release-audit 实际验收
- 技能路径：`.fox/skills/release-audit/SKILL.md`
- 按技能流程：read app.json → 核对 environment=development、port=8080、workers=4（全部通过）→ write 生成 **audit.md**（808 字节）→ 复读确认。
- audit.md 含三小节（检查对象、检查结果、注意事项），末尾独立一行 `AUDIT_LAB_V1`。
- 未修改 app.json。
- 注意事项已说明：仅检查开发环境；教学日志不称为生产测试；发布代号仅留在对话。

### 4. 技能方法解释（仅解释，不执行文件操作）
- 说明技能检查 app.json 的三项配置；实际验收时才 read+write audit.md，仅解释时不碰文件。
- 解释 Skill（任务方法与流程/标准）≠ Tool（read/write 等具体执行接口）。

### 5. release.log 教学数据读取
- 实际 read `release.log`，limit=200。
- 日志为教学数据（Training fixture），重复性说明：内容高度重复，每行格式一致，仅 `sample=000`~`sample=139` 编号递增；首行声明保留发布代号与仅限开发环境约束；均标注 synthetic/no production traffic。

## 关键约束（持续保留）
- 发布代号：**青石-29**
- 配置结果：environment=development、port=8080、workers=4
- 仅限 **development** 环境；发布代号不写入 app.json；教学数据不代表生产测量。

## 待办事项
- 无明确未完成的待办任务，所有用户请求均已完成。

### 6.3 再次从 JSONL 恢复，让真实模型检验保留信息

我们不直接继续使用压缩后的 Python 对象，而是重新读磁盘。这样同时验证 **压缩结果是否保存** 和 **模型能否依据摘要继续回答**。

暂时禁用工具，防止模型通过重新读取 app.json 补答案。下面的问题不再重复给出发布代号或配置数值。

In [15]:
compact_restored = make_harness(Session(JsonlSessionStorage(SESSION_FILE)), skills=loaded.skills, request_model=None)
attach(compact_restored, "after-compaction")
assert len(compact_restored.state.messages) == len(skilled.state.messages)
assert message_text(compact_restored.state.messages[0]).startswith("[Previous conversation summary]")
compact_restored.set_active_tools([])
answer_after_compaction = await run_step(compact_restored,
    "只根据已有上下文，不读取任何文件：本次发布代号是什么？当前端口和 workers 数量是什么？"
    "只允许修改哪个环境？说明教学日志是否能证明生产环境健康。请简短回答。", "06-压缩后恢复与回答")
assert "青石-29" in answer_after_compaction and "8080" in answer_after_compaction, (
    "真实模型未保留全部关键事实，这是摘要质量问题。请检查摘要并调整策略，不要硬编码回答。")
compact_restored.set_active_tools(["read", "write", "edit"])


[turn_start]
- 发布代号：青石-29
- 端口：8080；workers：4
- 仅允许修改：development 环境
- 教学日志为合成教学数据，不能证明生产环境健康。
[agent_end]

本步骤新增真实请求：1


**面试复盘**：压缩后上下文条数通常变少，历史条目反而增加一条。Session 是历史，模型 Context 是历史的一个工作视图。摘要质量并不由非空校验保证，需要像上面这样检查后续任务是否仍能正确完成。

## 7. 自动压缩：在每次模型请求前处理预算

手动压缩回答的是“我现在要求整理历史”。自动压缩回答的是“下一次请求之前，输入是否已经接近预算”。

下面从**手动压缩之前的 leaf** 分出独立会话，恢复那段较长的真实历史。然后降低给 Harness 的本地预算，使 `prepare_request` 在首次模型调用前触发压缩。

**这个预算仅在本地触发策略，不修改真实 API 服务的上下文上限。** 不伪造服务端溢出、不生成固定摘要，也不影响主会话。自动压缩应在本次 `prompt()` 内产生“summary 请求 → agent 请求”的顺序。

In [16]:
auto_session = compact_restored.session.fork(PRE_COMPACT_LEAF)
auto_history = auto_session.build_context()
history_estimate = estimate_context_tokens(auto_history)
assert history_estimate > OUTPUT_TOKENS, "当前历史过短，不能稳定展示自动压缩。"
AUTO_WINDOW = history_estimate + OUTPUT_TOKENS
AUTO_RESERVE = OUTPUT_TOKENS + 512
auto_model = model.model_copy(update={"context_window": AUTO_WINDOW})
auto_settings = CompactionSettings(enabled=True, reserve_tokens=AUTO_RESERVE, keep_recent_tokens=400)
auto = make_harness(auto_session, skills=loaded.skills, compaction=auto_settings, request_model=auto_model)
attach(auto, "auto-compaction")
auto.set_active_tools([])  # 本章只问历史，禁用工具以便观察纯粹的准备请求流程。

# 与 Harness 相同的估算组成：消息 + system prompt + 工具声明（这里是空数组）。
estimated_input = history_estimate + estimate_tokens(UserMessage(content=auto.state.system_prompt))
estimated_input += estimate_tokens(UserMessage(content=json.dumps([])))
threshold = AUTO_WINDOW - min(AUTO_RESERVE, AUTO_WINDOW // 2)
assert should_compact(estimated_input, AUTO_WINDOW, auto_settings)
table([{"历史估算": history_estimate, "含system的估算": estimated_input,
        "本地窗口": AUTO_WINDOW, "触发阈值": threshold, "保留预算": auto_settings.keep_recent_tokens}])

历史估算,含system的估算,本地窗口,触发阈值,保留预算
7247,7566,8447,6735,400


In [17]:
request_start, event_start = len(REQUESTS), len(EVENTS)
answer_auto = await run_step(auto,
    "只根据对话历史简短回答：本次发布代号、已修改的端口和 workers 数量，以及环境限制。不要读取文件。",
    "07-自动压缩与回答")
auto_requests = REQUESTS[request_start:]
auto_events = EVENTS[event_start:]
assert [r["kind"] for r in auto_requests] == ["summary", "agent"], "请检查自动压缩是否先于普通模型请求。"
assert any(e["type"] == "compaction_end" for e in auto_events)
assert "青石-29" in answer_auto and "8080" in answer_auto
table([{k: r.get(k) for k in ["number", "phase", "kind", "messages", "status", "seconds", "total"]}
       for r in auto_requests])
print("原始主会话仍可恢复；自动压缩只改变这个分支的上下文。")


[turn_start]

[compaction_start]

[compaction_end]
发布代号：青石-29。已修改：端口 8080，workers 4。限制：仅限 development 环境，不写入 app.json 的发布代号，日志为教学数据。
[agent_end]

本步骤新增真实请求：2


number,phase,kind,messages,status,seconds,total
23,07-自动压缩与回答,summary,1,stop,3.03,7170
24,07-自动压缩与回答,agent,3,stop,0.95,1095


原始主会话仍可恢复；自动压缩只改变这个分支的上下文。


**为什么是 prepare_request？** 首轮恢复出的历史就可能很长；后续 steering/follow-up 还会增加输入。每次请求前、待处理消息注入后的检查点，才能看到即将发送的内容。

**本地预算是估算，不是服务端 tokenizer。** 若摘要或最近消息仍太大，当前 Harness 会在触发压缩后的检查路径报预算错误。不要把继续降低阈值误认为能解决任意长输入；应考虑工具输出截断、分块摘要或减少输入。

## 8. 用量、证据和面试复盘

本表将普通回答请求和摘要请求分开计数。费用依赖服务端实际费率与缓存策略，本 Notebook 不用可能过期的价格估算金额；请结合服务商账单查看真实费用。

`total` 优先使用 provider 报告的 `total_tokens`，没有时才根据 fox_ai 的 input/output/cache 字段相加。前文“估算 token”与这里的实际 usage 是两个不同概念。

In [18]:
table([{k: r.get(k, "") for k in ["number", "phase", "kind", "messages", "status", "seconds", "input", "output", "total"]}
       for r in REQUESTS])
counts = Counter(r["kind"] for r in REQUESTS)
print("真实请求次数:", len(REQUESTS), "普通响应:", counts["agent"], "摘要:", counts["summary"])
print("provider 报告的累计 token:", sum(r.get("total", 0) for r in REQUESTS))

# 只保存脱敏的观测数据；不保存 options、密钥或 HTTP headers。
metrics_path = LAB_ROOT / "request_metrics.json"
public_rows = [{k: v for k, v in r.items() if k != "started"} for r in REQUESTS]
metrics_path.write_text(json.dumps(public_rows, ensure_ascii=False, indent=2), encoding="utf-8")
assert API_KEY not in metrics_path.read_text(encoding="utf-8")
assert API_KEY not in SESSION_FILE.read_text(encoding="utf-8")
print("会话与报告保留在:", LAB_ROOT)
print("请求统计:", metrics_path)

number,phase,kind,messages,status,seconds,input,output,total
1,01-普通响应,agent,1,stop,1.56,25,22,47
2,02-自定义工具,agent,1,toolUse,0.77,225,58,411
3,02-自定义工具,agent,3,stop,0.71,175,94,525
4,03-文件工具,agent,1,toolUse,0.56,684,44,728
5,03-文件工具,agent,3,toolUse,0.61,147,97,884
6,03-文件工具,agent,5,toolUse,0.72,142,99,1009
7,03-文件工具,agent,7,toolUse,0.69,139,69,1104
8,03-文件工具,agent,9,toolUse,0.82,139,81,1244
9,03-文件工具,agent,11,toolUse,0.91,135,79,1366
10,03-文件工具,agent,13,toolUse,0.58,257,37,1446


真实请求次数: 24 普通响应: 22 摘要: 2
provider 报告的累计 token: 53936
会话与报告保留在: C:\Users\Qin\AppData\Local\Temp\fox-agent-core-lab-9tr1uzmn
请求统计: C:\Users\Qin\AppData\Local\Temp\fox-agent-core-lab-9tr1uzmn\request_metrics.json


### 用实验回答这些问题

| 面试问题 | 本 Notebook 中的证据 |
| --- | --- |
| 一次 prompt 为什么有多个 turn？ | 第二章的 ToolCall / ToolResult / 最终回答 |
| 怎样证明工具真的执行了？ | 本地计算记录、修改后的 app.json、实际 audit.md |
| Agent 和 Harness 怎么分工？ | 第一章只有内存状态，第三章增加保存、策略和工具组合 |
| Session 是不是模型的上下文？ | 第六章历史条目增加，但发送的消息变少 |
| 恢复是不是重新执行历史工具？ | 第四章只重建消息，第六章关闭工具仍能回答历史问题 |
| Skill 是不是一种工具？ | 第五章正文提供方法，read/write 才产生副作用 |
| 压缩摘要是不是本地拼接字符串？ | 请求观测表中真实的 summary 调用及其 usage |
| 自动压缩发生在哪里？ | 第七章同一次 prompt 先 summary，再 agent |
| 能保证摘要永远不丢信息吗？ | 不能，所以压缩后还用真实提问验证关键事实 |

### 可继续尝试的练习

1. 改小 `keep_recent_tokens`，观察保留尾部与摘要质量的变化。
2. 把 Skill 的报告格式改成别的标记，重新加载后比较生成结果。
3. 用 `harness.follow_up(...)` 排队一条问题，观察它何时进入新一轮。注意已经结束的 Harness 需要再次 prompt 或 continue_ 才会处理排队消息。
4. 在另一个实验会话里尝试一个越界文件路径，观察 before Hook 返回的失败结果；不要修改这条主线已经保存的证据。
5. 比较逐条保存消息与仅在 run 结束时保存历史的恢复语义。

### 常见问题

- **找不到包**：从仓库内打开，选 Python 3.14 的项目内核，先运行第一格。修改源码后重启内核。
- **401/余额或限流错误**：检查选定服务对应的密钥与账户状态；Notebook 不会退回模拟模型。
- **没有工具调用**：确认模型和 API 支持 function calling，查看事件和模型输出；不能把直接文本回答当成工具执行成功。
- **达到轮次/总请求上限**：先检查是否重复操作，再调整预算。反复运行 live 单元会产生新的真实请求。
- **摘要为空、截断或没有保留事实**：查看 summary 请求状态与摘要，调整输出/保留预算；原始历史仍保留在 Session。
- **想复用上次会话文件**：使用输出中的 SESSION_FILE 路径创建新的 JsonlSessionStorage。再次执行准备目录单元会创建一个全新的实验目录。
- **Run All 的输出与截图不同**：真实模型行为有变化；本实验检查实际文件、调用记录和关键事实，不要求回复逐字一致。

本实验不会自动删除练习目录，方便查看 JSONL、Skill 和报告。确认不再需要后，可以手动删除打印出的 `fox-agent-core-lab-*` 目录。